In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# Decision Trees
Another method different than linear regression
### Requirements
Make sure to use a ML Cluster, for instance 12.2 LTS ML

## Load Dataset
Let's load the clean Airbnb dataset in again 
We created it in the previous notebook, it should exists in `/home/jovyan/work/outputs/airbnb/clean_data`

In [ ]:
file_path = f"/home/jovyan/work/outputs/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)

train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)

In [ ]:
from pyspark.ml.feature import StringIndexer

categorical_cols = <TODO>
index_output_cols = <TODO>

string_indexer = StringIndexer(inputCols=<TODO>, outputCols=<TODO>, handleInvalid="skip")

## VectorAssembler
Let's use the <a href="https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html?highlight=vectorassembler#pyspark.ml.feature.VectorAssembler" target="_blank">VectorAssembler</a> to combine all of our categorical and numeric inputs.

In [ ]:
from pyspark.ml.feature import VectorAssembler

# Filter for just numeric columns (and exclude price, the target column)
numeric_cols = <TODO>

# Combine output of StringIndexer defined above and numeric columns
assembler_inputs = <TODO>
vec_assembler = VectorAssembler(inputCols=<TODO>, outputCol=<TODO>)

## Decision Tree

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = <TODO>

## Training model with Pipeline
The following cell is expected to error, but we subsequently fix this.

In [ ]:
from pyspark.ml import Pipeline

# Combine stages into pipeline
stages = [<TODO>]
pipeline = Pipeline(stages=<TODO>)

# Train model with the train set
pipeline_model = pipeline.fit(<TODO>)

In [ ]:
assembler_inputs

In [ ]:
dt.setMaxBins(<TODO>)

In [ ]:
pipeline_model = pipeline.fit(<TODO>)

## Feature Importance

In [ ]:
dt_model = pipeline_model.stages[-1]
dt_model.show(truncate=False)

In [ ]:
dt_model.featureImportances

### Interpreting Feature Importance
It's complicated to interprete features by number, let's zip it with vec_assembler to name them

In [ ]:
import pandas as pd

features_df = pd.DataFrame(list(zip(vec_assembler.getInputCols(), dt_model.featureImportances)), columns=["feature", "importance"])
features_df

# Only a handful of features are > 0
this is because default **`maxDepth`** is 5, so there are only a few features that where considered

In [ ]:
top_n = 5

top_features = features_df.sort_values(["importance"], ascending=False)[:top_n]["feature"].values
print(top_features)

## Apply model to test set

In [ ]:
pred_df = pipeline_model.transform(<TODO>)

display(pred_df.select("features", "price", "prediction").orderBy("price", ascending=False))

In [ ]:
display(pred_df.select("features", "price", "prediction").orderBy("price", ascending=False).filter("prediction < 2000").filter("price <2000"))

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_evaluator = <TODO>

rmse = <TODO>
r2 = <TODO>
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")